# Task 3.2 — Error Analysis

This notebook analyzes system failures across three RAG configurations:
- `baseline_recursive`: Cosine similarity + recursive chunking
- `baseline_section`: Cosine similarity + section-aware chunking  
- `reranked_section`: Re-ranked + section-aware chunking

**Objectives:**
1. Identify 3 worst-performing queries and diagnose failure types
2. Identify 1 hallucination case and propose mitigation
3. Propose 2 actionable improvements

## Cell 1 — Setup & Load Data

In [ ]:
import json
import os
from collections import defaultdict, Counter
from difflib import SequenceMatcher

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/[3Q]Capstone1/Assignment3'

with open(f'{BASE_PATH}/task3_rag_output_baseline_recursive.json') as f:
    rec = json.load(f)
with open(f'{BASE_PATH}/task3_rag_output_baseline_section.json') as f:
    sec = json.load(f)
with open(f'{BASE_PATH}/task3_rag_output_reranked_section.json') as f:
    rerank = json.load(f)

print(f'baseline_recursive  : {rec["total"]} results | model={rec["model"]} | strategy={rec["strategy"]}')
print(f'baseline_section    : {sec["total"]} results | model={sec["model"]} | strategy={sec["strategy"]}')
print(f'reranked_section    : {rerank["total"]} results | model={rerank["model"]} | strategy={rerank["strategy"]}')

## Cell 2 — Performance Overview by Modality

In [ ]:
def similarity(a, b):
    """Answer-ground_truth text similarity as proxy for correctness."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def compute_stats(results, name):
    sims = [similarity(r['answer'], r['ground_truth']) for r in results]
    mod_sims = defaultdict(list)
    for r in results:
        mod_sims[r['modality']].append(similarity(r['answer'], r['ground_truth']))

    print(f'\n=== {name} ===')
    print(f'  Overall mean similarity : {sum(sims)/len(sims):.4f}')
    print(f'  Modality breakdown:')
    for mod in ['text', 'text-table', 'text-image', 'text-table-image']:
        if mod in mod_sims:
            scores = mod_sims[mod]
            print(f'    {mod:<20} mean={sum(scores)/len(scores):.4f}  (n={len(scores)})')
    return sims, mod_sims


rec_sims, rec_mod   = compute_stats(rec['results'],    'Baseline Recursive')
sec_sims, sec_mod   = compute_stats(sec['results'],    'Baseline Section')
rer_sims, rer_mod   = compute_stats(rerank['results'], 'Reranked Section')

print('\n📊 Key Observation:')
print('  text-only queries perform significantly better than multimodal queries')
print('  text-table-image shows the worst performance across all configurations')

## Cell 3 — Identify 3 Worst-Performing Queries

In [ ]:
# Score all results in baseline_recursive
scored = sorted(
    [{'sim': similarity(r['answer'], r['ground_truth']), **r} for r in rec['results']],
    key=lambda x: x['sim']
)

# Select 3 diverse failure cases:
# Case 1: text-image (multimodal — required)
# Case 2: text-table (table-dependent)
# Case 3: text-only

case1 = next(r for r in scored if r['modality'] == 'text-image'
             and 'biological pathway' in r['query'])
case2 = sorted(
    [r for r in scored if r['modality'] == 'text-table'],
    key=lambda x: x['sim']
)[0]
case3 = sorted(
    [r for r in scored if r['modality'] == 'text'],
    key=lambda x: x['sim']
)[0]

for label, case in [('CASE 1 (text-image)', case1), ('CASE 2 (text-table)', case2), ('CASE 3 (text)', case3)]:
    print(f'\n{"="*65}')
    print(f'❌ {label}  |  sim={case["sim"]:.4f}')
    print(f'{"="*65}')
    print(f'Query     : {case["query"]}')
    print(f'Modality  : {case["modality"]}')
    print(f'Answer    : {case["answer"][:300]}...')
    print(f'GT        : {case["ground_truth"][:300]}...')
    print(f'Retrieved : {[c["arxiv_id"] for c in case["retrieved_chunks"]]}') 

## Cell 4 — Failure Diagnosis

In [ ]:
print('=' * 65)
print('FAILURE DIAGNOSIS')
print('=' * 65)

print('''
CASE 1 — text-image query (biological pathway datasets)
  Query   : How do different biological pathway datasets compare in terms
             of distortion when using Euclidean vs mixed-curvature embeddings?
  Failure : RETRIEVAL ERROR (Multimodal)
  Diagnosis:
    - The correct paper (2401.15478v2) WAS retrieved (all 3 chunks from it)
    - However, the answer requires reading Figure 2(a) — a chart comparing
      distortion across datasets — which is an IMAGE the system cannot see
    - The text chunks contain only the caption [FIGURE: ...] placeholder,
      not the actual numerical values in the figure
    - System attempted to answer from the figure caption and hallucinated
      specific details ("points below t...") that were not in the text
  Failure Type: Retrieval error — image content not captured in index
''')

print('''
CASE 2 — text-table query (Sylber vs HuBERT performance)
  Query   : What are the key differences in performance between Sylber
             models and HuBERT base model across various speech tasks?
  Failure : RETRIEVAL ERROR (Table)
  Diagnosis:
    - Correct paper (2410.07168v2) was retrieved
    - Answer was "I don't have enough information" — system correctly
      refused to hallucinate
    - BUT the performance comparison data lives in a TABLE in the paper
    - During chunking, table was converted to Markdown but the specific
      numeric values (KS, IC, SID, SF, ASV scores) were likely split
      across chunk boundaries or lost in parsing
    - System could not find the quantitative comparison it needed
  Failure Type: Retrieval error — table content fragmented during chunking
''')

print('''
CASE 3 — text-only query (foreground class distribution)
  Query   : Does the foreground class distribution change between
             train and test sets?
  Failure : GENERATION ERROR
  Diagnosis:
    - Multiple papers retrieved (2410.11774v2, 2408.11208v3)
    - Ground truth answer is simply "Yes"
    - System retrieved partially relevant chunks but over-hedged:
      "The context does not provide specific information..."
    - The relevant sentence was present in the context but system
      failed to extract the simple factual answer from it
    - System prompt may not have been specific enough to encourage
      short, direct factual answers
  Failure Type: Generation error — over-cautious response to simple factual query
''')

## Cell 5 — Hallucination Case Analysis

In [ ]:
# Find hallucination: confident, detailed answer that diverges from ground truth
hall_case = next(
    r for r in rec['results']
    if 'media coverage' in r['query'].lower()
)

print('=' * 65)
print('🔴 HALLUCINATION CASE')
print('=' * 65)
print(f'Query        : {hall_case["query"]}')
print(f'Modality     : {hall_case["modality"]}')
print(f'Ground Truth : {hall_case["ground_truth"]}')
print(f'System Answer:\n{hall_case["answer"]}')

print()
print('--- Diagnosis ---')
print('''
Ground truth: "Yes" (a simple binary answer)

The system produced a long, confident answer with specific claims:
  - "significant spike in volume of news articles"
  - "market overreactions, characterized by anomalies in price movement"
  - "imbalance in representation of news coverage"

These specific details came from the retrieved context chunks (the paper
discusses hype-adjusted probability measures for NLP stock forecasting).
However, the question only asked for a yes/no answer. The system:
  1. Correctly identified that the answer is "Yes"
  2. Then OVER-GENERATED by adding context-derived details as if they
     were direct answers to the question
  3. Some specific numerical/causal claims may not be directly supported
     by the retrieved text, making this a hallucination via over-extension

Mitigation: Add explicit instruction in system prompt:
  "If the question can be answered with a short factual response (yes/no,
  a number, a name), provide that directly first before any elaboration.
  Do not extend your answer beyond what the context directly supports."
''')

## Cell 6 — Proposed Improvements

In [ ]:
print('=' * 65)
print('🔧 PROPOSED IMPROVEMENTS')
print('=' * 65)

print('''
IMPROVEMENT 1 — Multimodal Figure Captioning
─────────────────────────────────────────────
Problem identified: Cases 1 and most text-image failures occur because
figure content is stored only as [FIGURE: caption] placeholders in chunks.
The actual visual data (charts, diagrams, experimental results) is inaccessible.

Proposed fix:
  During ingestion (Task 1.1 extension), pass each extracted figure image
  through a multimodal LLM (e.g., GPT-4o Vision) to generate a detailed
  caption describing the visual content:
  
    "Figure 2(a): Bar chart comparing distortion scores across 5 biological
    pathway datasets. Mixed-curvature embeddings (blue bars) consistently
    show lower distortion than Euclidean embeddings (red bars) across all
    datasets including KEGG, Reactome, and WikiPathways..."

  Replace [FIGURE] placeholders with these rich captions before embedding.
  This directly addresses the root cause: the index does not contain the
  visual information needed to answer figure-dependent queries.

Expected impact: Significant improvement on text-image and text-table-image
queries (167 out of 496 = 33.7% of benchmark queries).
''')

print('''
IMPROVEMENT 2 — Table-Aware Chunking with Atomic Table Units
──────────────────────────────────────────────────────────────
Problem identified: Case 2 (text-table) shows the system retrieved the
correct paper but could not find the performance comparison table data.
Markdown tables are frequently split across chunk boundaries by the
recursive splitter when tables exceed 512 tokens.

Proposed fix:
  Extend the preprocessing step to detect Markdown table blocks
  (lines starting with | and ---) and treat them as atomic units,
  similar to the LaTeX formula protection already implemented:

    # Detect and protect tables before chunking  
    def protect_tables(text):
        table_pattern = re.compile(
            r'(\|.+\|\n\|[-:| ]+\|\n(?:\|.+\|\n)+)',
            re.MULTILINE
        )
        # Mark table boundaries so splitter never cuts mid-table
        return table_pattern.sub(
            lambda m: f'TABLE_START\n{m.group()}\nTABLE_END', text
        )

  Additionally, store each table as its own dedicated chunk with metadata
  tag {"content_type": "table"} to enable table-targeted retrieval.

Expected impact: Improvement on text-table queries (28 queries in benchmark).
Also benefits any query requiring quantitative comparisons from result tables.
''')

## Cell 7 — Performance Summary Table

In [ ]:
print('📊 Performance Summary by Configuration and Modality')
print()
print(f'{"Configuration":<25} {"Overall":>8} {"text":>8} {"text-table":>11} {"text-image":>11} {"text-t-i":>9}')
print('-' * 75)

configs = [
    ('Baseline Recursive',  rec['results']),
    ('Baseline Section',    sec['results']),
    ('Reranked Section',    rerank['results']),
]

for name, results in configs:
    sims      = [similarity(r['answer'], r['ground_truth']) for r in results]
    mod_sims  = defaultdict(list)
    for r in results:
        mod_sims[r['modality']].append(similarity(r['answer'], r['ground_truth']))

    overall = sum(sims)/len(sims)
    t       = sum(mod_sims['text'])/len(mod_sims['text'])
    tt      = sum(mod_sims['text-table'])/len(mod_sims['text-table'])
    ti      = sum(mod_sims['text-image'])/len(mod_sims['text-image'])
    tti     = sum(mod_sims['text-table-image'])/len(mod_sims['text-table-image'])

    print(f'{name:<25} {overall:>8.4f} {t:>8.4f} {tt:>11.4f} {ti:>11.4f} {tti:>9.4f}')

print()
print('Key findings:')
print('  1. Reranked section outperforms baseline on most modalities')
print('  2. text-only queries consistently outperform multimodal queries')
print('  3. text-table-image is the hardest modality across all configs')
print('  4. ~33.7% of queries (167/496) require multimodal understanding')